In [ ]:
import kagglehub, pandas as pd, os, re, warnings
warnings.filterwarnings('ignore')

path = kagglehub.dataset_download("nalgondalokesh/drug-food-interaction-dataset")
df = pd.read_csv(os.path.join(path, "drug_food_interactions.csv"))
print(f"✅ البيانات جاهزة: {df.shape[0]} صف")
df.head(3)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# تنظيف النص
df['clean'] = df['food_interactions'].fillna('').astype(str).str.lower()
df['clean'] = df['clean'].str.replace(r'[^a-z ]+', ' ', regex=True).str.strip()

# تصنيف بالكلمات المفتاحية
def label(t):
    t = str(t).lower()
    if not t.strip(): return 'No Interaction'
    if any(w in t for w in ['avoid','do not',"don't",'contraindicated','dangerous','toxic','fatal']): return 'Avoid'
    if any(w in t for w in ['increase','decrease','reduce','absorption','level','affect','alter']): return 'Monitor'
    if any(w in t for w in ['with food','with meal','with milk','with water','take with']): return 'Take With Food'
    if any(w in t for w in ['limit','moderate','caution','grapefruit','caffeine','alcohol']): return 'Limit'
    return 'General Interaction'

df['label'] = df['food_interactions'].apply(label)
print("📊 توزيع الفئات:")
print(df['label'].value_counts())
df['label'].value_counts().plot(kind='bar', color=['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6'])
plt.title('توزيع فئات التداخل الدوائي الغذائي')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['label'])
X = df['clean']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"🏋️ تدريب: {len(X_train)} | 🧪 اختبار: {len(X_test)}")

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ دقة النموذج: {acc:.2%}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
import re

def predict(drug_name, food_name):
    drug_rows = df[df['name'].str.lower() == drug_name.lower()]
    if drug_rows.empty:
        print("="*50)
        print(f"💊 الدواء: {drug_name} | 🍽️ الأكل: {food_name}")
        print(f"⚪ الدواء غير موجود في الداتاسيت")
        print("="*50); return 'No Data'
    interaction_text = drug_rows['food_interactions'].fillna('').values[0]
    if not interaction_text.strip():
        print("="*50)
        print(f"💊 الدواء: {drug_name} | 🍽️ الأكل: {food_name}")
        print(f"⚪ No Interaction - لا تداخل موثّق")
        print("="*50); return 'No Interaction'
    food_mentioned = food_name.lower() in interaction_text.lower()
    clean = re.sub(r'[^a-z ]+', ' ', interaction_text.lower()).strip()
    pred_enc = model.predict([clean])[0]
    pred_label = le.inverse_transform([pred_enc])[0]
    proba = model.predict_proba([clean])[0].max()
    icons = {'Avoid':'🔴','Monitor':'🟡','Take With Food':'🟢','Limit':'🟠','General Interaction':'🔵','No Interaction':'⚪'}
    print("="*50)
    print(f"💊 الدواء: {drug_name} | 🍽️ الأكل: {food_name}")
    if food_mentioned:
        print(f"⚠️  {food_name} مذكور في تحذيرات الدواء!")
        print(f"🔮 {icons.get(pred_label,'❓')} {pred_label} | 📊 الثقة: {proba:.1%}")
    else:
        print(f"✅ {food_name} غير مذكور في التحذيرات")
        print(f"📋 التداخل الموثّق: {interaction_text[:100]}...")
        print(f"🔮 التصنيف العام: {icons.get(pred_label,'❓')} {pred_label}")
    print("="*50); return pred_label

# اختبارات
predict("atorvastatin", "orange")
print()
predict("atorvastatin", "grapefruit")
print()
predict("warfarin", "spinach")